In [3]:
from google.colab import drive
drive.mount('/content/drive')
data_root='/content/drive/My Drive/ChatBot'

Mounted at /content/drive


In [4]:
import json
from flask import Flask, render_template, request
import string
import random
import nltk
import numpy as np
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from keras.api._v2.keras import Sequential
from keras.api._v2.keras.layers import Dense, Dropout
nltk.download("punkt")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
data_file=open(data_root + '/intents.json').read()
data=json.loads(data_file)

In [6]:
words = [] #For Bow model/ vocabulary for patterns

classes = [] #For Bow model/ vocabulary for tags

data_x = [] #For storing each pattern

data_y = [] #For storing tag corresponding to each pattern in data_X C

#Iterating over all the intents

for intent in data["intents"]:

    for pattern in intent["patterns"]:

        tokens = nltk.word_tokenize (pattern) # tokenize each pattern

        words.extend(tokens) #and append tokens to words

        data_x.append(pattern) #appending pattern to data_X


        data_y.append(intent["tag"]), #appending the associated tag to each pattern

        #adding the tag to the classes if it's not there already
        if intent["tag"] not in classes:
           classes.append(intent["tag"])

#initializing lemmatizer to get stem of words

lemmatizer = WordNetLemmatizer()

#lemmatize all the words in the vocab and convert them to lowercase

#if the words don't appear in punctuation

words = [lemmatizer.lemmatize (word.lower()) for word in words if word not in string.punctuation]

#sorting the vocab and classes in alphabetical order and taking the # set to ensure no duplicates occur

words = sorted(set(words))

classes = sorted(set(classes))

In [7]:
training = []

out_empty = [0] * len(classes)

#creating the bag of words model
for idx, doc in enumerate(data_x):

      bow = []

      text = lemmatizer.lemmatize(doc.lower())

      for word in words:

            bow.append(1) if word in text else bow.append(0)

            #mark the index of class that the current pattern is associated

            #to

      output_row = list(out_empty)

      output_row[classes.index(data_y[idx])] = 1

#add the one hot encoded Bow and associated classes to training
      training.append([bow, output_row])

#shuffle the data and convert it to an array

random.shuffle (training)

training = np.array(training, dtype=object)

#split the features and target labels

train_x = np.array(list(training[:,0]))
train_y = np.array(list(training[:,1]))

In [8]:
model=Sequential()
model.add(Dense (128, input_shape=(len(train_x[0]),), activation="relu"))
model.add(Dropout (0.5))
model.add(Dense (64, activation="relu"))
model.add(Dropout (0.5))
model.add(Dense(len(train_y[0]), activation="softmax"))
adam = tf.keras.optimizers.legacy.Adam(learning_rate=0.01, decay=1e-6)
model.compile(loss='categorical_crossentropy',optimizer=adam,metrics=["accuracy"])
print(model.summary())
model.fit(x=train_x, y=train_y, epochs=300, verbose=1)




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               24448     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 45)                2925      
                                                                 
Total params: 35629 (139.18 KB)
Trainable params: 35629 (139.18 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None
Epoch 1/300
4

In [9]:
def clean_text(text):
  tokens = nltk.word_tokenize(text)
  tokens = [lemmatizer.lemmatize (word) for word in tokens]
  return tokens

def bag_of_words(text, vocab):
  tokens= clean_text(text)
  bow=[0]*len(vocab)
  for w in tokens:
    for idx, word in enumerate(vocab):
      if word == w:
        bow[idx] = 1
  return np.array(bow)

def pred_class(text, vocab, labels):
  bow = bag_of_words(text, vocab)
  result = model.predict(np.array([bow]))[0] #Extracting probabilities
  thresh = 0.5
  y_pred = [[indx, res] for indx, res in enumerate(result) if res > thresh]
  y_pred.sort(key=lambda x: x[1], reverse=True) #sorting by values of probability in decreasing order
  return_list = []
  for r in y_pred:
    return_list.append(labels[r[0]]) #Contains labels (tags) for highest probability
  return return_list

def get_response(intents_list, intents_json):
  if len(intents_list) == 0:
    result = "Sorry! I don't understand."
  else:
    tag = intents_list[0]
    list_of_intents=intents_json["intents"]
    for i in list_of_intents:
      if i["tag"] == tag:
        result = random.choice(i["responses"])
        break
    return result



In [ ]:
print("Press 0 if you don't want to chat with our ChatBot.")
while True:
  message = input("")
  if message == "0":
    break
  intents = pred_class(message,words,classes)
  result = get_response(intents,data)
  print(result)

Press 0 if you don't want to chat with our ChatBot.
1/1 [==============================] - 0s 136ms/step
Good morning!
1/1 [==============================] - 0s 23ms/step
Very good and you?
1/1 [==============================] - 0s 24ms/step
That is perfect!
1/1 [==============================] - 0s 20ms/step
what do you need?
1/1 [==============================] - 0s 20ms/step
CPUs, or Central Processing Units, are the brains of your computer. They handle instructions and calculations.
1/1 [==============================] - 0s 30ms/step
Common RAM types include DDR4. Consider options like Corsair Vengeance LPX.
1/1 [==============================] - 0s 23ms/step
I can help you verify compatibility. Please provide the CPU and motherboard models.
